In [15]:
from openai import OpenAI
import pandas as pd
import minsearch


## Ingestion

In [16]:
df = pd.read_csv('../data/cleaned_data.csv')

In [17]:

documents = df.to_dict(orient='records')

In [18]:
index = minsearch.Index(['task', 'category', 'difficulty', 'duration_estimate'
       'framework_name', 'reasoning', 'instructions', 'tags'],
        keyword_fields=["id"])


In [24]:
index.fit(documents)

print(documents[0])

{'task': 'Write a 2-page project summary', 'category': 'work', 'difficulty': 'medium', 'duration_estimate': 45, 'framework_name': 'Time Blocking', 'reasoning': 'Time Blocking helps allocate a clear writing window.', 'instructions': 'Block 45 minutes, outline key points, write summary, revise.', 'tags': 'work;writing;planning'}


In [20]:
q = "How can I organize my workspace efficiently using a structured method?"

## RAG flow

In [ ]:
client = OpenAI()

def search(query):
    boost = {}

    results = index.search(
        query=query,
        filter_dict={},
        boost_dict=boost,
        num_results=10
    )

    return results

In [23]:
index.search(q, num_results=10)

[{'task': 'Clean bedroom nightstand',
  'category': 'home',
  'difficulty': 'easy',
  'duration_estimate': 15,
  'framework_name': 'Task Batching',
  'reasoning': 'Batching cleaning tasks increases efficiency.',
  'instructions': 'Wipe surface, organize items, clean drawer.',
  'tags': 'home;cleaning'},
 {'task': 'Organize digital notes',
  'category': 'home',
  'difficulty': 'medium',
  'duration_estimate': 35,
  'framework_name': 'GTD',
  'reasoning': 'GTD helps categorize digital clutter.',
  'instructions': 'Sort notes, create folders, delete old items.',
  'tags': 'home;digital'},
 {'task': 'Clean laundry area',
  'category': 'home',
  'difficulty': 'medium',
  'duration_estimate': 25,
  'framework_name': 'Task Batching',
  'reasoning': 'Batching cleaning tasks increases efficiency.',
  'instructions': 'Wipe surfaces, organize detergents, clean floor.',
  'tags': 'home;cleaning'},
 {'task': 'Clean living room shelves',
  'category': 'home',
  'difficulty': 'easy',
  'duration_esti

In [ ]:


prompt_template = """
You're a productivity advisor. Answer the QUESTION based on the CONTEXT from our productivity tasks dataset.
Use only the facts from the CONTEXT when answering the QUESTION.

QUESTION: {question}

CONTEXT:
{context}
""".strip()

entry_template = """
task: {task}
category: {category}
difficulty: {difficulty}
duration_estimate: {duration_estimate}
instructions: {instructions}
reasoning: {reasoning}
tags: {tags}
""".strip()

def build_prompt(query, search_results):
    context = ""

    for doc in search_results:
        context += entry_template.format(**doc) + "\n\n"

    prompt = prompt_template.format(question=query, context=context).strip()
    return prompt



[{'task': 'Clean bedroom nightstand', 'category': 'home', 'difficulty': 'easy', 'duration_estimate': 15, 'framework_name': 'Task Batching', 'reasoning': 'Batching cleaning tasks increases efficiency.', 'instructions': 'Wipe surface, organize items, clean drawer.', 'tags': 'home;cleaning'}, {'task': 'Organize digital notes', 'category': 'home', 'difficulty': 'medium', 'duration_estimate': 35, 'framework_name': 'GTD', 'reasoning': 'GTD helps categorize digital clutter.', 'instructions': 'Sort notes, create folders, delete old items.', 'tags': 'home;digital'}, {'task': 'Clean laundry area', 'category': 'home', 'difficulty': 'medium', 'duration_estimate': 25, 'framework_name': 'Task Batching', 'reasoning': 'Batching cleaning tasks increases efficiency.', 'instructions': 'Wipe surfaces, organize detergents, clean floor.', 'tags': 'home;cleaning'}, {'task': 'Clean living room shelves', 'category': 'home', 'difficulty': 'easy', 'duration_estimate': 20, 'framework_name': 'Task Batching', 'reas

In [26]:
search_results = search(q)
prompt = build_prompt(q, search_results)


In [28]:
model='gpt-4o-mini'
def llm(prompt, model=model):
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}]
    )
    
    return response.choices[0].message.content


In [ ]:
def rag(query, model=model):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(prompt)
    return answer

In [29]:
question = "How can I organize my workspace efficiently using a structured method?"
answer = rag(question)
print(answer)

To organize your workspace efficiently using a structured method, you can follow these steps based on the principles outlined in the context:

1. **Clear the Space**: Start by clearing your desk or workspace to create a fresh and uncluttered environment.

2. **Sort Items**: Go through the items on your desk and sort them into categories. Identify what you need frequently and what can be stored away.

3. **Batch Similar Tasks**: For example, if you're organizing different areas, consider batching similar tasks together, such as cleaning and organizing different parts of your workspace like drawers or shelves. This approach enhances efficiency.

4. **Use a Categorization Method**: Apply the Getting Things Done (GTD) method to categorize your items. This can help clarify what is essential and prioritize what you need at hand.

5. **Label and Store**: Once you’ve sorted items, consider labeling containers or storage solutions to easily identify contents later, much like organizing craft su

In [ ]:
df_question = pd.read_csv('../data/ground-truth-retrieval.csv')

In [9]:
from pydantic import BaseModel, Field
from typing import List

class Task(BaseModel):
    task: str = Field(description="Name or title of the task")
    category: str = Field(description="Category such as work, home, fitness, study, etc.")
    difficulty: str = Field(description="Difficulty level: easy, medium, hard")
    duration_estimate: int = Field(description="Estimated duration in minutes")
    instructions: str = Field(description="Step-by-step instructions for completing the task")
    reasoning: str = Field(description="Explanation of why this task maps to a specific framework")
    tags: str = Field(description="Comma-separated tags for search and filtering")


In [10]:
class TaskDataset(BaseModel):
    tasks: List[Task]

In [12]:
openai_client = OpenAI()
prompt = """
Generate a dataset of 50 diverse productivity tasks.
Cover different categories such as home, work, fitness, study, personal, and creative.
Vary difficulty levels and duration estimates.
Each task must include: task, category, difficulty, duration_estimate, instructions, reasoning, tags.
Return the dataset in the exact schema provided.
""".strip()
response = openai_client.responses.parse(
    model="gpt-5.4-mini",
    input=[{"role": "user", "content": prompt}],
    text_format=TaskDataset,
)

dataset = response.output_parsed
df = pd.DataFrame([task.model_dump() for task in dataset.tasks])
df.to_csv("../data/tasks.csv", index=False)
print(f"Generated {len(df)} tasks")


Generated 53 tasks


In [13]:
q = "Find a task that teaches me how to improve my focus using a structured method."

In [14]:
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": q}]
)
response.choices[0].message.content 

"Improving focus is a valuable skill that can be cultivated through a structured method. Here’s a task that utilizes the Pomodoro Technique, a popular time management method designed to enhance concentration and productivity.\n\n### Task: Implement the Pomodoro Technique\n\n**Objective:** To improve focus and productivity through time management.\n\n#### Step 1: Prepare Your Workspace\n- **Choose a quiet environment** free of distractions. \n- **Remove any clutter** from your desk or workspace.\n- **Gather necessary materials**, such as paper, pens, or digital tools you will need for your task.\n\n#### Step 2: Select a Task\n- Choose a specific task or project you want to focus on. Ensure it's something that can be accomplished in segments.\n\n#### Step 3: Set Up a Timer\n- Use a timer, phone app, or an online Pomodoro timer.\n- **Set the timer for 25 minutes.** This is one Pomodoro session.\n\n#### Step 4: Work on the Task\n- Start working on your chosen task as soon as the timer star

In [ ]:


# Load dataset
df = pd.read_csv("../data/data.csv")   # notebook is in sibling folder

# Convert rows to dicts
documents = df.to_dict(orient="records")

# Build MinSearch index
index = Index(
    text_fields=[
        "task",
        "instructions",
        "reasoning",
        "tags",
        "category",
        "difficulty"
    ],
    keyword_fields=[
        "duration_estimate"
    ]
)

# Fit index
index.fit(documents)
